# O2 Inference From Raw CSV

Самодостаточный ноутбук. На вход подаются только:

- `daimler_component_properties.csv`
- `daimler_mixtures_train.csv`
- `daimler_mixtures_test.csv`

Ноутбук внутри себя:
- выполняет `transform_daimler_synergy_features`,
- строит `O2`-признаки,
- запускает `hierarchical_model/train_hierarchical_model.py` через встроенный `main()`,
- сохраняет `prediction.csv`.

Ноутбук не обращается к внешним `.py`-скриптам во время выполнения.


In [ ]:
TRANSFORM_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport math\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\n\nR_GAS_CONSTANT = 8.314\n\nTRAIN_TARGET_COLS = [\n    "Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %",\n    "Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm",\n]\n\nGLOBAL_COLS = [\n    "Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C",\n    "Время испытания | - Daimler Oxidation Test (DOT), ч",\n    "Количество биотоплива | - Daimler Oxidation Test (DOT), % масс",\n    "Дозировка катализатора, категория",\n]\n\nMIXTURE_KEY_COLS = [\n    "scenario_id",\n    "Компонент",\n    "Наименование партии",\n]\n\nREQUIRED_MIXTURE_COLS = MIXTURE_KEY_COLS + [\n    "Массовая доля, %",\n    *GLOBAL_COLS,\n]\n\nPROPS_REQUIRED_COLS = [\n    "Компонент",\n    "Наименование партии",\n    "Наименование показателя",\n    "Значение показателя",\n]\n\nPROP_TYPE_AO = "Тип АО"\nPROP_ACTIVE_NO = "Активный Азот / Кислород, % масс. (N или O)"\nPROP_MO = "% масс. (Mo)"\nPROP_ZN = "Массовая доля цинка, ASTM D6481"\nPROP_BORON = "Содержание Бора"\nPROP_S = "Массовая доля серы, ASTM D6481"\nPROP_CA = "Массовая доля кальция, ASTM D6481"\nPROP_CA_MG = "Содержание металла (Ca/Mg), % масс."\nPROP_E_BOND = "Энергия диссоциации связи Х-Н, ккал/моль"\nPROP_IONIZATION = "Потенциал ионизации,эВ"\nPROP_STERIC = "Стерический фактор, Å3"\nPROP_CHEM_POT = "Химический потенциал, Дж/моль"\nPROP_HOMO = "Энергия ВЗМО, эВ"\nPROP_LUMO = "Энергия НСМО, эВ"\n\nAO_TYPE_PHENOL = "фенол"\nAO_TYPE_DPA = "дифениламин"\n\nPROPERTY_NAME_ALIASES = {\n    "массовая доля цинка, astm d6481": PROP_ZN,\n    "массовая доля цинка | astm d6481": PROP_ZN,\n    "массовая доля серы, astm d6481": PROP_S,\n    "массовая доля серы | astm d6481": PROP_S,\n    "массовая доля кальция, astm d6481": PROP_CA,\n    "массовая доля кальция | astm d6481": PROP_CA,\n    "% масс. mo": PROP_MO,\n    "% масс. (mo)": PROP_MO,\n    "энергия взмо эв": PROP_HOMO,\n    "энергия взмо, эв": PROP_HOMO,\n}\n\n\ndef normalize_string(value: Any) -> str:\n    if pd.isna(value):\n        return ""\n    return re.sub(r"\\s+", " ", str(value).strip())\n\n\ndef normalize_casefold(value: Any) -> str:\n    return normalize_string(value).casefold()\n\n\ndef normalize_property_name(value: Any) -> str:\n    normalized = normalize_string(value)\n    return PROPERTY_NAME_ALIASES.get(normalized.casefold(), normalized)\n\n\ndef normalize_property_value(value: Any) -> Any:\n    if pd.isna(value):\n        return np.nan\n\n    normalized = normalize_string(value)\n    if not normalized:\n        return np.nan\n\n    numeric_value = pd.to_numeric(normalized, errors="coerce")\n    if pd.notna(numeric_value):\n        return float(numeric_value)\n\n    return normalized\n\n\ndef assert_required_columns(df: pd.DataFrame, required: list[str], df_name: str) -> None:\n    missing = [col for col in required if col not in df.columns]\n    if missing:\n        raise ValueError(f"В {df_name} отсутствуют обязательные колонки: {missing}")\n\n\ndef read_csv_strict(path: Path) -> pd.DataFrame:\n    if not path.exists():\n        raise FileNotFoundError(f"Файл не найден: {path}")\n    return pd.read_csv(path)\n\n\ndef component_is(component_name: str, prefix: str) -> bool:\n    return normalize_string(component_name).startswith(prefix)\n\n\ndef build_property_tables(props_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:\n    exact_df = props_df[props_df["Наименование партии"].map(normalize_casefold) != "typical"].copy()\n    typical_df = props_df[props_df["Наименование партии"].map(normalize_casefold) == "typical"].copy()\n\n    wide_exact = exact_df.pivot_table(\n        index=["Компонент", "Наименование партии"],\n        columns="Наименование показателя",\n        values="Значение показателя",\n        aggfunc="first",\n    ).reset_index()\n\n    wide_typical = typical_df.pivot_table(\n        index=["Компонент"],\n        columns="Наименование показателя",\n        values="Значение показателя",\n        aggfunc="first",\n    ).reset_index()\n\n    property_columns = sorted(\n        (set(wide_exact.columns) - {"Компонент", "Наименование партии"})\n        | (set(wide_typical.columns) - {"Компонент"})\n    )\n\n    for col in property_columns:\n        if col not in wide_exact.columns:\n            wide_exact[col] = np.nan\n        if col not in wide_typical.columns:\n            wide_typical[col] = np.nan\n\n    wide_exact = wide_exact[["Компонент", "Наименование партии"] + property_columns]\n    wide_typical = wide_typical[["Компонент"] + property_columns]\n\n    return wide_exact, wide_typical, property_columns\n\n\ndef enrich_mixture_with_properties(\n    mixture_df: pd.DataFrame,\n    wide_exact: pd.DataFrame,\n    wide_typical: pd.DataFrame,\n    property_columns: list[str],\n) -> pd.DataFrame:\n    enriched = mixture_df.merge(\n        wide_exact,\n        on=["Компонент", "Наименование партии"],\n        how="left",\n    )\n    enriched = enriched.merge(\n        wide_typical,\n        on=["Компонент"],\n        how="left",\n        suffixes=("", "__typ"),\n    )\n\n    for col in property_columns:\n        typ_col = f"{col}__typ"\n        enriched[col] = enriched[col].where(enriched[col].notna(), enriched[typ_col])\n\n    drop_cols = [f"{col}__typ" for col in property_columns]\n    enriched = enriched.drop(columns=drop_cols)\n\n    return enriched\n\n\ndef safe_series(df: pd.DataFrame, col: str) -> pd.Series:\n    if col not in df.columns:\n        return pd.Series(dtype=float)\n    return pd.to_numeric(df[col], errors="coerce")\n\n\ndef sum_product_cross(a: pd.Series, b: pd.Series) -> float:\n    a_clean = pd.to_numeric(a, errors="coerce").dropna()\n    b_clean = pd.to_numeric(b, errors="coerce").dropna()\n    if a_clean.empty or b_clean.empty:\n        return 0.0\n    return float(a_clean.sum() * b_clean.sum())\n\n\ndef calculate_k(e_value: float, a_value: float, temperature_c: float) -> float:\n    temperature_k = float(temperature_c) + 273.0\n    return float(\n        float(a_value) * math.exp(-(4184.0 * float(e_value)) / (R_GAS_CONSTANT * temperature_k))\n    )\n\n\ndef scenario_features_strict(scenario_df: pd.DataFrame, is_train: bool) -> dict[str, Any]:\n    row: dict[str, Any] = {\n        "scenario_id": scenario_df["scenario_id"].iloc[0],\n        GLOBAL_COLS[0]: scenario_df[GLOBAL_COLS[0]].iloc[0],\n        GLOBAL_COLS[1]: scenario_df[GLOBAL_COLS[1]].iloc[0],\n        GLOBAL_COLS[2]: scenario_df[GLOBAL_COLS[2]].iloc[0],\n        GLOBAL_COLS[3]: scenario_df[GLOBAL_COLS[3]].iloc[0],\n    }\n\n    if is_train:\n        row[TRAIN_TARGET_COLS[0]] = scenario_df[TRAIN_TARGET_COLS[0]].iloc[0]\n        row[TRAIN_TARGET_COLS[1]] = scenario_df[TRAIN_TARGET_COLS[1]].iloc[0]\n\n    temperature_c = float(scenario_df[GLOBAL_COLS[0]].iloc[0])\n\n    ao_df = scenario_df[scenario_df["Компонент"].map(lambda x: component_is(x, "Антиоксидант"))].copy()\n    dispersant_df = scenario_df[scenario_df["Компонент"].map(lambda x: component_is(x, "Дисперсант"))].copy()\n    molybdenum_df = scenario_df[\n        scenario_df["Компонент"].map(lambda x: component_is(x, "Соединение_молибдена"))\n    ].copy()\n    antiwear_df = scenario_df[\n        scenario_df["Компонент"].map(lambda x: component_is(x, "Противоизносная_присадка"))\n    ].copy()\n\n    ao_synergy_value = 0.0\n    if PROP_TYPE_AO in ao_df.columns and PROP_ACTIVE_NO in ao_df.columns:\n        ao_work = ao_df[[PROP_TYPE_AO, PROP_ACTIVE_NO]].copy()\n        ao_work[PROP_TYPE_AO] = ao_work[PROP_TYPE_AO].map(normalize_casefold)\n        ao_work[PROP_ACTIVE_NO] = pd.to_numeric(ao_work[PROP_ACTIVE_NO], errors="coerce")\n\n        phenol_values = ao_work.loc[ao_work[PROP_TYPE_AO] == AO_TYPE_PHENOL, PROP_ACTIVE_NO].dropna()\n        dpa_values = ao_work.loc[ao_work[PROP_TYPE_AO] == AO_TYPE_DPA, PROP_ACTIVE_NO].dropna()\n\n        ao_synergy_value = sum_product_cross(dpa_values, phenol_values)\n\n    row["synergy_ao_phenol_x_diphenylamine_active_no"] = ao_synergy_value\n\n    dpa_active_values = pd.Series(dtype=float)\n    if PROP_TYPE_AO in ao_df.columns and PROP_ACTIVE_NO in ao_df.columns:\n        ao_work = ao_df[[PROP_TYPE_AO, PROP_ACTIVE_NO]].copy()\n        ao_work[PROP_TYPE_AO] = ao_work[PROP_TYPE_AO].map(normalize_casefold)\n        ao_work[PROP_ACTIVE_NO] = pd.to_numeric(ao_work[PROP_ACTIVE_NO], errors="coerce")\n        dpa_active_values = ao_work.loc[ao_work[PROP_TYPE_AO] == AO_TYPE_DPA, PROP_ACTIVE_NO].dropna()\n\n    mo_values = safe_series(molybdenum_df, PROP_MO).dropna()\n    row["synergy_ao_diphenylamine_active_no_x_mo"] = sum_product_cross(dpa_active_values, mo_values)\n\n    zinc_values = safe_series(scenario_df, PROP_ZN).dropna()\n    boron_values = safe_series(dispersant_df, PROP_BORON)\n    boron_values = boron_values[boron_values != 0].dropna()\n    row["synergy_zn_x_boron_dispersant"] = sum_product_cross(zinc_values, boron_values)\n\n    sulfur_sum = float(safe_series(antiwear_df, PROP_S).dropna().sum())\n    metal_sum = float(\n        safe_series(scenario_df, PROP_CA).dropna().sum()\n        + safe_series(scenario_df, PROP_CA_MG).dropna().sum()\n    )\n    row["synergy_aw_sulfur_x_ca_ca_mg"] = float(sulfur_sum * metal_sum)\n\n    ao_k_values: list[float] = []\n    ao_k_weighted_terms: list[float] = []\n\n    if PROP_E_BOND in ao_df.columns and PROP_STERIC in ao_df.columns:\n        e_values = pd.to_numeric(ao_df[PROP_E_BOND], errors="coerce")\n        steric_values = pd.to_numeric(ao_df[PROP_STERIC], errors="coerce")\n\n        if PROP_ACTIVE_NO in ao_df.columns:\n            active_no_values = pd.to_numeric(ao_df[PROP_ACTIVE_NO], errors="coerce").fillna(0.0)\n        else:\n            active_no_values = pd.Series(0.0, index=ao_df.index)\n\n        valid_mask = e_values.notna() & steric_values.notna()\n\n        for idx in ao_df.index[valid_mask]:\n            k_value = calculate_k(\n                e_value=float(e_values.loc[idx]),\n                a_value=float(steric_values.loc[idx]),\n                temperature_c=temperature_c,\n            )\n            ao_k_values.append(k_value)\n            ao_k_weighted_terms.append(k_value * float(active_no_values.loc[idx]))\n\n    if len(ao_k_values) == 1:\n        row["ao_k_avg_weighted_active_no"] = float(ao_k_values[0])\n        row["ao_k_avg_arithmetic"] = float(ao_k_values[0])\n    elif ao_k_values:\n        row["ao_k_avg_weighted_active_no"] = float(np.sum(ao_k_weighted_terms))\n        row["ao_k_avg_arithmetic"] = float(np.mean(ao_k_values))\n    else:\n        row["ao_k_avg_weighted_active_no"] = 0.0\n        row["ao_k_avg_arithmetic"] = 0.0\n\n    ionization_values = safe_series(ao_df, PROP_IONIZATION).dropna()\n    row["ao_ionization_min"] = float(ionization_values.min()) if not ionization_values.empty else np.nan\n\n    homo_values = safe_series(ao_df, PROP_HOMO).dropna()\n    row["ao_homo_max"] = float(homo_values.max()) if not homo_values.empty else np.nan\n\n    return row\n\n\ndef build_component_level_output(enriched_df: pd.DataFrame) -> pd.DataFrame:\n    component_df = enriched_df.copy()\n\n    drop_cols = [\n        PROP_CHEM_POT,\n        PROP_LUMO,\n        PROP_E_BOND,\n        PROP_STERIC,\n    ]\n    existing_drop_cols = [col for col in drop_cols if col in component_df.columns]\n    component_df = component_df.drop(columns=existing_drop_cols)\n\n    return component_df\n\n\ndef build_scenario_level_output(enriched_df: pd.DataFrame, is_train: bool) -> pd.DataFrame:\n    rows = [\n        scenario_features_strict(group, is_train=is_train)\n        for _, group in enriched_df.groupby("scenario_id", sort=True)\n    ]\n    return pd.DataFrame(rows).sort_values("scenario_id").reset_index(drop=True)\n\n\ndef transform_dataset(\n    mixture_df: pd.DataFrame,\n    wide_exact: pd.DataFrame,\n    wide_typical: pd.DataFrame,\n    property_columns: list[str],\n    is_train: bool,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    enriched = enrich_mixture_with_properties(\n        mixture_df=mixture_df,\n        wide_exact=wide_exact,\n        wide_typical=wide_typical,\n        property_columns=property_columns,\n    )\n\n    component_level = build_component_level_output(enriched)\n    scenario_level = build_scenario_level_output(enriched, is_train=is_train)\n\n    return component_level, scenario_level\n'


In [ ]:
from typing import Any
import re
import pandas as pd

ID_COL = "scenario_id"
COMPONENT_COL = "Компонент"
WEIGHT_COL = "Массовая доля, %"
AO_TYPE_COL = "Тип АО"
ACTIVE_NO_COL = "Активный Азот / Кислород, % масс. (N или O)"
SUBSTRATE_COL = "Класс субстрата"
TBN_ASTM_COL = "Щелочное число, ASTM D2896"
TBN_GOST_COL = "Щелочное число, ГОСТ 11362"


def normalize_string_o2(value: Any) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def parse_numeric_like_o2(value: Any) -> float:
    if pd.isna(value):
        return float("nan")
    text = normalize_string_o2(value)
    if not text:
        return float("nan")
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    text = text.replace("%", "").replace(" ", "")
    if text.count(",") == 1 and text.count(".") == 0:
        text = text.replace(",", ".")
    match = re.search(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:e[+-]?\d+)?", text.casefold())
    if not match:
        return float("nan")
    try:
        return float(match.group(0))
    except ValueError:
        return float("nan")


def series_numeric_o2(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(dtype=float)
    return df[col].map(parse_numeric_like_o2)


def weighted_mean_o2(values: pd.Series, weights: pd.Series) -> float:
    mask = values.notna() & weights.notna()
    if not mask.any():
        return 0.0
    v = values[mask].astype(float)
    w = weights[mask].astype(float)
    weight_sum = float(w.sum())
    if weight_sum <= 0.0:
        return float(v.mean())
    return float((v * w).sum() / weight_sum)


def build_o2_features(component_df: pd.DataFrame, scenario_df: pd.DataFrame) -> pd.DataFrame:
    component_groups = {
        scenario_id: group.reset_index(drop=True)
        for scenario_id, group in component_df.groupby(ID_COL, sort=False)
    }
    rows: list[dict[str, Any]] = []

    for _, scenario_row in scenario_df.iterrows():
        scenario_id = scenario_row[ID_COL]
        group = component_groups[scenario_id]
        component_names = group[COMPONENT_COL].map(normalize_string_o2)

        ao_df = group[component_names.map(lambda x: x.startswith("Антиоксидант"))].copy()
        detergent_df = group[component_names.map(lambda x: x.startswith("Детергент"))].copy()

        ao_types = ao_df.get(AO_TYPE_COL, pd.Series(index=ao_df.index, dtype=object)).map(normalize_string_o2)
        ao_active = series_numeric_o2(ao_df, ACTIVE_NO_COL).fillna(0.0)
        det_substrate = detergent_df.get(SUBSTRATE_COL, pd.Series(index=detergent_df.index, dtype=object)).map(normalize_string_o2)
        det_weights = series_numeric_o2(detergent_df, WEIGHT_COL).fillna(0.0)
        det_tbn = series_numeric_o2(detergent_df, TBN_ASTM_COL)
        det_tbn = det_tbn.where(det_tbn.notna(), series_numeric_o2(detergent_df, TBN_GOST_COL))

        salicylate_mask = det_substrate.str.contains("Салицилат", case=False, na=False)
        salicylate_tbn = weighted_mean_o2(det_tbn[salicylate_mask], det_weights[salicylate_mask])
        ca_salicylate_present = float(det_substrate.str.contains("Салицилат кальция", case=False, na=False).any())
        mg_detergent_present = float(det_substrate.str.contains("магния", case=False, na=False).any())
        dpa_active = float(ao_active[ao_types == "Дифениламин"].sum())
        phenol_active = float(ao_active[ao_types == "Фенол"].sum())

        rows.append(
            {
                ID_COL: scenario_id,
                "o2_salicylate_tbn_x_amine_ao": salicylate_tbn * dpa_active,
                "o2_salicylate_tbn_x_phenol_ao": salicylate_tbn * phenol_active,
                "o2_salicylate_tbn_x_amine_x_phenol": salicylate_tbn * dpa_active * phenol_active,
                "o2_ca_salicylate_present": ca_salicylate_present,
                "o2_mg_detergent_present": mg_detergent_present,
            }
        )

    return pd.DataFrame(rows)


def merge_features(base_df: pd.DataFrame, feature_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    merged = base_df.merge(feature_df[[ID_COL] + columns], on=ID_COL, how="left")
    merged[columns] = merged[columns].fillna(0.0)
    return merged


In [ ]:
TRAINER_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport copy\nimport json\nimport math\nimport random\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.model_selection import train_test_split\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\n\n\nID_COL = "scenario_id"\nCOMPONENT_NAME_COL = "Компонент"\n\nTARGET_COLS = [\n    "Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %",\n    "Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm",\n]\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef read_csv_strict(path: Path) -> pd.DataFrame:\n    if not path.exists():\n        raise FileNotFoundError(f"Файл не найден: {path}")\n    return pd.read_csv(path)\n\n\ndef validate_required_columns(df: pd.DataFrame, required_columns: list[str], df_name: str) -> None:\n    missing_columns = [column for column in required_columns if column not in df.columns]\n    if missing_columns:\n        raise ValueError(f"В {df_name} отсутствуют обязательные колонки: {missing_columns}")\n\n\ndef normalize_string(value: Any) -> str:\n    if pd.isna(value):\n        return ""\n    return re.sub(r"\\s+", " ", str(value).strip())\n\n\ndef extract_component_family(component_name: Any) -> str:\n    normalized_name = normalize_string(component_name)\n    if not normalized_name:\n        return "__MISSING_FAMILY__"\n\n    match = re.match(r"^(.*)_\\d+$", normalized_name)\n    if match:\n        return match.group(1)\n\n    return normalized_name\n\n\ndef parse_numeric_like(value: Any) -> float:\n    if pd.isna(value):\n        return float("nan")\n\n    text = normalize_string(value)\n    if not text:\n        return float("nan")\n\n    lowered = text.casefold()\n    if lowered in {"нет", "none", "nan"}:\n        return float("nan")\n\n    if re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}(?: \\d{2}:\\d{2}:\\d{2})?", text):\n        return float("nan")\n\n    cleaned = text.replace("−", "-").replace("–", "-").replace("—", "-")\n    cleaned = cleaned.replace("≤", "").replace("≥", "").replace("<", "").replace(">", "").replace("~", "")\n    cleaned = cleaned.replace("%", "").replace(" ", "")\n    cleaned = cleaned.replace("°C", "").replace("°c", "").replace("⁰С", "").replace("⁰с", "")\n\n    if cleaned.count(",") == 1 and cleaned.count(".") == 0:\n        cleaned = cleaned.replace(",", ".")\n    elif cleaned.count(",") > 1 and cleaned.count(".") == 0:\n        return float("nan")\n    elif cleaned.count(",") >= 1 and cleaned.count(".") >= 1:\n        return float("nan")\n\n    numeric_pattern = r"^[+-]?(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:e[+-]?\\d+)?$"\n    if not re.fullmatch(numeric_pattern, cleaned.casefold()):\n        return float("nan")\n\n    return float(cleaned)\n\n\ndef detect_component_feature_columns(\n    component_train_df: pd.DataFrame,\n    scenario_train_df: pd.DataFrame,\n) -> tuple[list[str], list[str], list[str]]:\n    repeated_scenario_columns = [\n        column\n        for column in scenario_train_df.columns\n        if column in component_train_df.columns and column not in {ID_COL, *TARGET_COLS}\n    ]\n\n    excluded_columns = {\n        ID_COL,\n        *TARGET_COLS,\n        *repeated_scenario_columns,\n    }\n\n    base_component_columns = [\n        column for column in component_train_df.columns if column not in excluded_columns\n    ]\n\n    numeric_columns = component_train_df[base_component_columns].select_dtypes(include=["number"]).columns.tolist()\n    object_columns = [column for column in base_component_columns if column not in numeric_columns]\n\n    auto_numeric_columns: list[str] = []\n    categorical_columns: list[str] = []\n\n    for column in object_columns:\n        if column == COMPONENT_NAME_COL:\n            categorical_columns.append(column)\n            continue\n\n        parsed_series = component_train_df[column].map(parse_numeric_like)\n        non_null_mask = component_train_df[column].map(lambda value: normalize_string(value) != "")\n        non_null_count = int(non_null_mask.sum())\n\n        if non_null_count == 0:\n            continue\n\n        parsed_success_ratio = float(parsed_series.notna().sum()) / float(non_null_count)\n\n        if parsed_success_ratio >= 0.8:\n            auto_numeric_columns.append(column)\n            continue\n\n        unique_count = component_train_df[column].fillna("").astype(str).nunique()\n        if unique_count <= 64:\n            categorical_columns.append(column)\n\n    return numeric_columns, auto_numeric_columns, categorical_columns\n\n\ndef build_category_vocabulary(values: pd.Series) -> dict[str, int]:\n    unique_values = sorted({normalize_string(value) for value in values if normalize_string(value)})\n    vocabulary = {\n        "__PAD__": 0,\n        "__UNK__": 1,\n        "__MISSING__": 2,\n    }\n    for index, value in enumerate(unique_values, start=3):\n        vocabulary[value] = index\n    return vocabulary\n\n\ndef build_feature_spec(\n    component_train_df: pd.DataFrame,\n    scenario_train_df: pd.DataFrame,\n) -> dict[str, Any]:\n    validate_required_columns(component_train_df, [ID_COL, COMPONENT_NAME_COL], "component_train")\n    validate_required_columns(scenario_train_df, [ID_COL, *TARGET_COLS], "scenario_train")\n\n    numeric_columns, auto_numeric_columns, categorical_columns = detect_component_feature_columns(\n        component_train_df=component_train_df,\n        scenario_train_df=scenario_train_df,\n    )\n\n    numeric_arrays: list[np.ndarray] = []\n    selected_numeric_columns: list[str] = []\n    selected_auto_numeric_columns: list[str] = []\n\n    for column in numeric_columns:\n        values = pd.to_numeric(component_train_df[column], errors="coerce").to_numpy(dtype=np.float32)\n        if np.isfinite(values).any():\n            numeric_arrays.append(values.reshape(-1, 1))\n            selected_numeric_columns.append(column)\n\n    for column in auto_numeric_columns:\n        values = component_train_df[column].map(parse_numeric_like).to_numpy(dtype=np.float32)\n        if np.isfinite(values).any():\n            numeric_arrays.append(values.reshape(-1, 1))\n            selected_auto_numeric_columns.append(column)\n\n    if not numeric_arrays:\n        raise ValueError("После фильтрации не осталось числовых component-level признаков.")\n\n    component_numeric_matrix = np.concatenate(numeric_arrays, axis=1).astype(np.float32)\n    component_numeric_mean = np.nanmean(component_numeric_matrix, axis=0)\n    component_numeric_std = np.nanstd(component_numeric_matrix, axis=0)\n\n    component_numeric_mean = np.where(np.isnan(component_numeric_mean), 0.0, component_numeric_mean)\n    component_numeric_std = np.where(\n        np.isnan(component_numeric_std) | (component_numeric_std < 1e-8),\n        1.0,\n        component_numeric_std,\n    )\n\n    global_columns = [\n        column for column in scenario_train_df.columns if column not in {ID_COL, *TARGET_COLS}\n    ]\n    if not global_columns:\n        raise ValueError("В scenario_train не осталось global признаков.")\n\n    global_matrix = scenario_train_df[global_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)\n    global_mean = np.nanmean(global_matrix, axis=0)\n    global_std = np.nanstd(global_matrix, axis=0)\n\n    global_mean = np.where(np.isnan(global_mean), 0.0, global_mean)\n    global_std = np.where(np.isnan(global_std) | (global_std < 1e-8), 1.0, global_std)\n\n    target_matrix = scenario_train_df[TARGET_COLS].to_numpy(dtype=np.float32)\n    target_mean = np.mean(target_matrix, axis=0)\n    target_std = np.std(target_matrix, axis=0)\n    target_std = np.where(target_std < 1e-8, 1.0, target_std)\n\n    category_vocabularies: dict[str, dict[str, int]] = {}\n    for column in categorical_columns:\n        category_vocabularies[column] = build_category_vocabulary(component_train_df[column])\n\n    family_series = component_train_df[COMPONENT_NAME_COL].map(extract_component_family)\n    family_vocabulary = build_category_vocabulary(family_series)\n\n    return {\n        "component_numeric_columns": selected_numeric_columns,\n        "component_auto_numeric_columns": selected_auto_numeric_columns,\n        "component_categorical_columns": categorical_columns,\n        "global_columns": global_columns,\n        "category_vocabularies": category_vocabularies,\n        "family_vocabulary": family_vocabulary,\n        "component_numeric_mean": component_numeric_mean.astype(np.float32),\n        "component_numeric_std": component_numeric_std.astype(np.float32),\n        "global_mean": global_mean.astype(np.float32),\n        "global_std": global_std.astype(np.float32),\n        "target_mean": target_mean.astype(np.float32),\n        "target_std": target_std.astype(np.float32),\n    }\n\n\ndef encode_component_numeric(component_df: pd.DataFrame, feature_spec: dict[str, Any]) -> np.ndarray:\n    numeric_arrays: list[np.ndarray] = []\n\n    for column in feature_spec["component_numeric_columns"]:\n        values = pd.to_numeric(component_df[column], errors="coerce").to_numpy(dtype=np.float32)\n        numeric_arrays.append(values.reshape(-1, 1))\n\n    for column in feature_spec["component_auto_numeric_columns"]:\n        values = component_df[column].map(parse_numeric_like).to_numpy(dtype=np.float32)\n        numeric_arrays.append(values.reshape(-1, 1))\n\n    numeric_matrix = np.concatenate(numeric_arrays, axis=1).astype(np.float32)\n    numeric_matrix = np.where(\n        np.isnan(numeric_matrix),\n        feature_spec["component_numeric_mean"].reshape(1, -1),\n        numeric_matrix,\n    )\n    numeric_matrix = (\n        numeric_matrix - feature_spec["component_numeric_mean"].reshape(1, -1)\n    ) / feature_spec["component_numeric_std"].reshape(1, -1)\n\n    return numeric_matrix.astype(np.float32)\n\n\ndef encode_global_features(scenario_row: pd.Series, feature_spec: dict[str, Any]) -> np.ndarray:\n    global_values = pd.to_numeric(\n        scenario_row[feature_spec["global_columns"]],\n        errors="coerce",\n    ).to_numpy(dtype=np.float32)\n\n    global_values = np.where(np.isnan(global_values), feature_spec["global_mean"], global_values)\n    global_values = (global_values - feature_spec["global_mean"]) / feature_spec["global_std"]\n    return global_values.astype(np.float32)\n\n\ndef encode_categorical_feature(values: pd.Series, vocabulary: dict[str, int]) -> np.ndarray:\n    encoded: list[int] = []\n    for value in values.map(normalize_string):\n        if not value:\n            encoded.append(vocabulary["__MISSING__"])\n        else:\n            encoded.append(vocabulary.get(value, vocabulary["__UNK__"]))\n    return np.asarray(encoded, dtype=np.int64)\n\n\ndef encode_family_ids(component_names: pd.Series, family_vocabulary: dict[str, int]) -> np.ndarray:\n    encoded: list[int] = []\n    for family_name in component_names.map(extract_component_family):\n        if not family_name:\n            encoded.append(family_vocabulary["__MISSING__"])\n        else:\n            encoded.append(family_vocabulary.get(family_name, family_vocabulary["__UNK__"]))\n    return np.asarray(encoded, dtype=np.int64)\n\n\nclass ScenarioHierarchicalDataset(Dataset):\n    def __init__(\n        self,\n        component_df: pd.DataFrame,\n        scenario_df: pd.DataFrame,\n        feature_spec: dict[str, Any],\n        is_train: bool,\n    ) -> None:\n        component_groups = {\n            scenario_id: group.reset_index(drop=True)\n            for scenario_id, group in component_df.groupby(ID_COL, sort=False)\n        }\n\n        self.samples: list[dict[str, Any]] = []\n\n        for _, scenario_row in scenario_df.iterrows():\n            scenario_id = scenario_row[ID_COL]\n            if scenario_id not in component_groups:\n                raise ValueError(f"Для scenario_id={scenario_id} нет component-level строк.")\n\n            component_rows = component_groups[scenario_id]\n\n            sample = {\n                "scenario_id": scenario_id,\n                "component_numeric": encode_component_numeric(component_rows, feature_spec),\n                "component_categorical": {\n                    column: encode_categorical_feature(\n                        component_rows[column],\n                        feature_spec["category_vocabularies"][column],\n                    )\n                    for column in feature_spec["component_categorical_columns"]\n                },\n                "family_ids": encode_family_ids(\n                    component_rows[COMPONENT_NAME_COL],\n                    feature_spec["family_vocabulary"],\n                ),\n                "global_features": encode_global_features(scenario_row, feature_spec),\n            }\n\n            if is_train:\n                targets = scenario_row[TARGET_COLS].to_numpy(dtype=np.float32)\n                targets = (targets - feature_spec["target_mean"]) / feature_spec["target_std"]\n                sample["targets"] = targets.astype(np.float32)\n\n            self.samples.append(sample)\n\n    def __len__(self) -> int:\n        return len(self.samples)\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        return self.samples[index]\n\n\ndef make_collate_fn(feature_spec: dict[str, Any], is_train: bool):\n    numeric_dim = len(feature_spec["component_numeric_mean"])\n    global_dim = len(feature_spec["global_columns"])\n    categorical_columns = feature_spec["component_categorical_columns"]\n\n    def collate_fn(batch: list[dict[str, Any]]) -> dict[str, Any]:\n        batch_size = len(batch)\n        max_components = max(sample["component_numeric"].shape[0] for sample in batch)\n\n        component_numeric = torch.zeros((batch_size, max_components, numeric_dim), dtype=torch.float32)\n        component_mask = torch.zeros((batch_size, max_components), dtype=torch.bool)\n        family_ids = torch.zeros((batch_size, max_components), dtype=torch.long)\n        global_features = torch.zeros((batch_size, global_dim), dtype=torch.float32)\n\n        component_categorical = {\n            column: torch.zeros((batch_size, max_components), dtype=torch.long)\n            for column in categorical_columns\n        }\n\n        if is_train:\n            targets = torch.zeros((batch_size, len(TARGET_COLS)), dtype=torch.float32)\n\n        scenario_ids: list[str] = []\n\n        for batch_index, sample in enumerate(batch):\n            component_count = sample["component_numeric"].shape[0]\n\n            component_numeric[batch_index, :component_count] = torch.from_numpy(sample["component_numeric"])\n            component_mask[batch_index, :component_count] = True\n            family_ids[batch_index, :component_count] = torch.from_numpy(sample["family_ids"])\n            global_features[batch_index] = torch.from_numpy(sample["global_features"])\n\n            for column in categorical_columns:\n                component_categorical[column][batch_index, :component_count] = torch.from_numpy(\n                    sample["component_categorical"][column]\n                )\n\n            if is_train:\n                targets[batch_index] = torch.from_numpy(sample["targets"])\n\n            scenario_ids.append(sample["scenario_id"])\n\n        output = {\n            "scenario_id": scenario_ids,\n            "component_numeric": component_numeric,\n            "component_mask": component_mask,\n            "family_ids": family_ids,\n            "component_categorical": component_categorical,\n            "global_features": global_features,\n        }\n\n        if is_train:\n            output["targets"] = targets\n\n        return output\n\n    return collate_fn\n\n\nclass HierarchicalScenarioRegressor(nn.Module):\n    def __init__(\n        self,\n        component_numeric_dim: int,\n        global_dim: int,\n        categorical_cardinalities: dict[str, int],\n        family_cardinality: int,\n        model_dim: int = 64,\n        dropout: float = 0.15,\n    ) -> None:\n        super().__init__()\n\n        self.family_cardinality = family_cardinality\n        self.model_dim = model_dim\n\n        self.component_numeric_encoder = nn.Sequential(\n            nn.Linear(component_numeric_dim, model_dim),\n            nn.ReLU(),\n            nn.LayerNorm(model_dim),\n            nn.Dropout(dropout),\n        )\n\n        self.component_categorical_embeddings = nn.ModuleDict()\n        for column, cardinality in categorical_cardinalities.items():\n            embedding_dim = min(16, max(4, int(math.sqrt(cardinality)) + 1))\n            self.component_categorical_embeddings[column] = nn.Sequential(\n                nn.Embedding(cardinality, embedding_dim, padding_idx=0),\n                nn.Linear(embedding_dim, model_dim, bias=False),\n            )\n\n        self.family_embedding = nn.Embedding(family_cardinality, model_dim, padding_idx=0)\n\n        self.component_post_encoder = nn.Sequential(\n            nn.Linear(model_dim, model_dim),\n            nn.ReLU(),\n            nn.LayerNorm(model_dim),\n            nn.Dropout(dropout),\n        )\n\n        self.family_encoder = nn.Sequential(\n            nn.Linear(model_dim * 2, model_dim),\n            nn.ReLU(),\n            nn.LayerNorm(model_dim),\n            nn.Dropout(dropout),\n        )\n\n        self.global_encoder = nn.Sequential(\n            nn.Linear(global_dim, model_dim),\n            nn.ReLU(),\n            nn.LayerNorm(model_dim),\n            nn.Dropout(dropout),\n        )\n\n        self.family_score_mlp = nn.Sequential(\n            nn.Linear(model_dim * 2, model_dim),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(model_dim, 1),\n        )\n\n        self.head = nn.Sequential(\n            nn.Linear(model_dim * 3, 128),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(128, 64),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(64, len(TARGET_COLS)),\n        )\n\n    def forward(\n        self,\n        component_numeric: torch.Tensor,\n        component_categorical: dict[str, torch.Tensor],\n        family_ids: torch.Tensor,\n        component_mask: torch.Tensor,\n        global_features: torch.Tensor,\n    ) -> torch.Tensor:\n        component_states = self.component_numeric_encoder(component_numeric)\n\n        for column, embedding_stack in self.component_categorical_embeddings.items():\n            component_states = component_states + embedding_stack(component_categorical[column])\n\n        component_states = self.component_post_encoder(component_states)\n        component_states = component_states * component_mask.unsqueeze(-1)\n\n        family_one_hot = F.one_hot(\n            family_ids.clamp(min=0),\n            num_classes=self.family_cardinality,\n        ).float()\n\n        family_one_hot = family_one_hot * component_mask.unsqueeze(-1).float()\n\n        family_sum = torch.einsum("bnf,bnd->bfd", family_one_hot, component_states)\n        family_count = family_one_hot.sum(dim=1)\n\n        family_mask = family_count > 0\n        family_count_safe = family_count.clamp_min(1e-6).unsqueeze(-1)\n        family_mean = family_sum / family_count_safe\n\n        family_index_tensor = torch.arange(\n            self.family_cardinality,\n            device=component_states.device,\n            dtype=torch.long,\n        ).unsqueeze(0).expand(component_states.shape[0], -1)\n\n        family_emb = self.family_embedding(family_index_tensor)\n        family_states = self.family_encoder(torch.cat([family_mean, family_emb], dim=-1))\n        family_states = family_states * family_mask.unsqueeze(-1)\n\n        global_state = self.global_encoder(global_features)\n        global_expanded = global_state.unsqueeze(1).expand(-1, self.family_cardinality, -1)\n\n        family_scores = self.family_score_mlp(torch.cat([family_states, global_expanded], dim=-1)).squeeze(-1)\n        family_scores = family_scores.masked_fill(~family_mask, float("-inf"))\n\n        family_weights = torch.softmax(family_scores, dim=1)\n        family_weights = torch.where(\n            family_mask,\n            family_weights,\n            torch.zeros_like(family_weights),\n        )\n\n        weighted_family = (family_states * family_weights.unsqueeze(-1)).sum(dim=1)\n\n        masked_family_states = family_states.masked_fill(~family_mask.unsqueeze(-1), float("-inf"))\n        max_family = masked_family_states.max(dim=1).values\n        max_family = torch.where(torch.isfinite(max_family), max_family, torch.zeros_like(max_family))\n\n        final_features = torch.cat([weighted_family, max_family, global_state], dim=-1)\n        return self.head(final_features)\n\n\ndef inverse_scale_targets(values: np.ndarray, feature_spec: dict[str, Any]) -> np.ndarray:\n    return values * feature_spec["target_std"].reshape(1, -1) + feature_spec["target_mean"].reshape(1, -1)\n\n\ndef calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, Any]:\n    metrics: dict[str, Any] = {"per_target": {}}\n    rmse_values: list[float] = []\n    mae_values: list[float] = []\n    r2_values: list[float] = []\n\n    for index, target_name in enumerate(TARGET_COLS):\n        rmse_value = float(np.sqrt(mean_squared_error(y_true[:, index], y_pred[:, index])))\n        mae_value = float(mean_absolute_error(y_true[:, index], y_pred[:, index]))\n        r2_value = float(r2_score(y_true[:, index], y_pred[:, index]))\n\n        metrics["per_target"][target_name] = {\n            "rmse": rmse_value,\n            "mae": mae_value,\n            "r2": r2_value,\n        }\n\n        rmse_values.append(rmse_value)\n        mae_values.append(mae_value)\n        r2_values.append(r2_value)\n\n    metrics["mean_rmse"] = float(np.mean(rmse_values))\n    metrics["mean_mae"] = float(np.mean(mae_values))\n    metrics["mean_r2"] = float(np.mean(r2_values))\n    return metrics\n\n\ndef evaluate_model(\n    model: nn.Module,\n    dataloader: DataLoader,\n    device: torch.device,\n    feature_spec: dict[str, Any],\n) -> tuple[dict[str, Any], pd.DataFrame]:\n    model.eval()\n\n    scenario_ids: list[str] = []\n    target_batches: list[np.ndarray] = []\n    prediction_batches: list[np.ndarray] = []\n\n    with torch.no_grad():\n        for batch in dataloader:\n            predictions = model(\n                component_numeric=batch["component_numeric"].to(device),\n                component_categorical={key: value.to(device) for key, value in batch["component_categorical"].items()},\n                family_ids=batch["family_ids"].to(device),\n                component_mask=batch["component_mask"].to(device),\n                global_features=batch["global_features"].to(device),\n            )\n            target_batches.append(batch["targets"].cpu().numpy())\n            prediction_batches.append(predictions.cpu().numpy())\n            scenario_ids.extend(batch["scenario_id"])\n\n    scaled_targets = np.concatenate(target_batches, axis=0)\n    scaled_predictions = np.concatenate(prediction_batches, axis=0)\n\n    targets = inverse_scale_targets(scaled_targets, feature_spec)\n    predictions = inverse_scale_targets(scaled_predictions, feature_spec)\n\n    metrics = calculate_metrics(targets, predictions)\n\n    output_df = pd.DataFrame({ID_COL: scenario_ids})\n    for index, target_name in enumerate(TARGET_COLS):\n        output_df[target_name] = targets[:, index]\n        output_df[f"pred::{target_name}"] = predictions[:, index]\n\n    return metrics, output_df\n\n\ndef predict_for_test(\n    model: nn.Module,\n    dataloader: DataLoader,\n    device: torch.device,\n    feature_spec: dict[str, Any],\n) -> pd.DataFrame:\n    model.eval()\n\n    scenario_ids: list[str] = []\n    prediction_batches: list[np.ndarray] = []\n\n    with torch.no_grad():\n        for batch in dataloader:\n            predictions = model(\n                component_numeric=batch["component_numeric"].to(device),\n                component_categorical={key: value.to(device) for key, value in batch["component_categorical"].items()},\n                family_ids=batch["family_ids"].to(device),\n                component_mask=batch["component_mask"].to(device),\n                global_features=batch["global_features"].to(device),\n            )\n            prediction_batches.append(predictions.cpu().numpy())\n            scenario_ids.extend(batch["scenario_id"])\n\n    scaled_predictions = np.concatenate(prediction_batches, axis=0)\n    predictions = inverse_scale_targets(scaled_predictions, feature_spec)\n\n    output_df = pd.DataFrame({ID_COL: scenario_ids})\n    for index, target_name in enumerate(TARGET_COLS):\n        output_df[target_name] = predictions[:, index]\n\n    return output_df\n\n\ndef save_json(data: dict[str, Any], path: Path) -> None:\n    with path.open("w", encoding="utf-8") as file:\n        json.dump(data, file, ensure_ascii=False, indent=2)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Иерархическая модель для transformed Daimler DOT данных.")\n    parser.add_argument("--component-train", required=True, type=Path)\n    parser.add_argument("--component-test", required=True, type=Path)\n    parser.add_argument("--scenario-train", required=True, type=Path)\n    parser.add_argument("--scenario-test", required=True, type=Path)\n    parser.add_argument("--out-dir", required=True, type=Path)\n    parser.add_argument("--epochs", default=150, type=int)\n    parser.add_argument("--batch-size", default=8, type=int)\n    parser.add_argument("--learning-rate", default=1e-3, type=float)\n    parser.add_argument("--weight-decay", default=1e-4, type=float)\n    parser.add_argument("--patience", default=30, type=int)\n    parser.add_argument("--val-size", default=0.2, type=float)\n    parser.add_argument("--seed", default=42, type=int)\n    args = parser.parse_args()\n\n    if not 0.0 < args.val_size < 1.0:\n        raise ValueError("--val-size должен быть в диапазоне (0, 1).")\n\n    set_seed(args.seed)\n\n    out_dir = args.out_dir\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    component_train_df = read_csv_strict(args.component_train)\n    component_test_df = read_csv_strict(args.component_test)\n    scenario_train_df = read_csv_strict(args.scenario_train)\n    scenario_test_df = read_csv_strict(args.scenario_test)\n\n    validate_required_columns(component_train_df, [ID_COL, COMPONENT_NAME_COL], "component_train")\n    validate_required_columns(component_test_df, [ID_COL, COMPONENT_NAME_COL], "component_test")\n    validate_required_columns(scenario_train_df, [ID_COL, *TARGET_COLS], "scenario_train")\n    validate_required_columns(scenario_test_df, [ID_COL], "scenario_test")\n\n    train_ids, valid_ids = train_test_split(\n        scenario_train_df[ID_COL].tolist(),\n        test_size=args.val_size,\n        random_state=args.seed,\n    )\n\n    train_component_split_df = component_train_df[\n        component_train_df[ID_COL].isin(train_ids)\n    ].copy().reset_index(drop=True)\n    valid_component_split_df = component_train_df[\n        component_train_df[ID_COL].isin(valid_ids)\n    ].copy().reset_index(drop=True)\n\n    train_scenario_split_df = scenario_train_df[\n        scenario_train_df[ID_COL].isin(train_ids)\n    ].copy().reset_index(drop=True)\n    valid_scenario_split_df = scenario_train_df[\n        scenario_train_df[ID_COL].isin(valid_ids)\n    ].copy().reset_index(drop=True)\n\n    feature_spec = build_feature_spec(\n        component_train_df=train_component_split_df,\n        scenario_train_df=train_scenario_split_df,\n    )\n\n    train_dataset = ScenarioHierarchicalDataset(\n        component_df=train_component_split_df,\n        scenario_df=train_scenario_split_df,\n        feature_spec=feature_spec,\n        is_train=True,\n    )\n    valid_dataset = ScenarioHierarchicalDataset(\n        component_df=valid_component_split_df,\n        scenario_df=valid_scenario_split_df,\n        feature_spec=feature_spec,\n        is_train=True,\n    )\n    test_dataset = ScenarioHierarchicalDataset(\n        component_df=component_test_df,\n        scenario_df=scenario_test_df,\n        feature_spec=feature_spec,\n        is_train=False,\n    )\n\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=args.batch_size,\n        shuffle=True,\n        collate_fn=make_collate_fn(feature_spec, is_train=True),\n    )\n    valid_loader = DataLoader(\n        valid_dataset,\n        batch_size=args.batch_size,\n        shuffle=False,\n        collate_fn=make_collate_fn(feature_spec, is_train=True),\n    )\n    test_loader = DataLoader(\n        test_dataset,\n        batch_size=args.batch_size,\n        shuffle=False,\n        collate_fn=make_collate_fn(feature_spec, is_train=False),\n    )\n\n    categorical_cardinalities = {\n        column: len(vocabulary)\n        for column, vocabulary in feature_spec["category_vocabularies"].items()\n    }\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n    model = HierarchicalScenarioRegressor(\n        component_numeric_dim=len(feature_spec["component_numeric_mean"]),\n        global_dim=len(feature_spec["global_columns"]),\n        categorical_cardinalities=categorical_cardinalities,\n        family_cardinality=len(feature_spec["family_vocabulary"]),\n        model_dim=64,\n        dropout=0.15,\n    ).to(device)\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=args.learning_rate,\n        weight_decay=args.weight_decay,\n    )\n    loss_fn = nn.SmoothL1Loss(beta=1.0)\n\n    best_state: dict[str, torch.Tensor] | None = None\n    best_metrics: dict[str, Any] | None = None\n    best_epoch = -1\n    best_score = float("inf")\n    no_improvement_epochs = 0\n    history: list[dict[str, Any]] = []\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        running_loss = 0.0\n        sample_count = 0\n\n        for batch in train_loader:\n            optimizer.zero_grad()\n\n            predictions = model(\n                component_numeric=batch["component_numeric"].to(device),\n                component_categorical={key: value.to(device) for key, value in batch["component_categorical"].items()},\n                family_ids=batch["family_ids"].to(device),\n                component_mask=batch["component_mask"].to(device),\n                global_features=batch["global_features"].to(device),\n            )\n            targets = batch["targets"].to(device)\n\n            loss = loss_fn(predictions, targets)\n            if not torch.isfinite(loss):\n                raise RuntimeError("Loss стал NaN/Inf. Проверьте данные и масштабирование.")\n\n            loss.backward()\n            optimizer.step()\n\n            batch_size = targets.shape[0]\n            running_loss += float(loss.item()) * batch_size\n            sample_count += batch_size\n\n        train_loss = running_loss / max(sample_count, 1)\n        valid_metrics, _ = evaluate_model(model, valid_loader, device, feature_spec)\n\n        history.append(\n            {\n                "epoch": epoch,\n                "train_loss": train_loss,\n                "valid_mean_rmse": valid_metrics["mean_rmse"],\n                "valid_mean_mae": valid_metrics["mean_mae"],\n                "valid_mean_r2": valid_metrics["mean_r2"],\n            }\n        )\n\n        print(\n            f"epoch={epoch} "\n            f"train_loss={train_loss:.6f} "\n            f"valid_mean_rmse={valid_metrics[\'mean_rmse\']:.6f} "\n            f"valid_mean_r2={valid_metrics[\'mean_r2\']:.6f}"\n        )\n\n        if valid_metrics["mean_rmse"] < best_score:\n            best_score = valid_metrics["mean_rmse"]\n            best_metrics = valid_metrics\n            best_state = copy.deepcopy(model.state_dict())\n            best_epoch = epoch\n            no_improvement_epochs = 0\n        else:\n            no_improvement_epochs += 1\n\n        if no_improvement_epochs >= args.patience:\n            break\n\n    if best_state is None or best_metrics is None:\n        raise RuntimeError("Не удалось сохранить лучшую модель.")\n\n    model.load_state_dict(best_state)\n\n    final_valid_metrics, valid_predictions_df = evaluate_model(model, valid_loader, device, feature_spec)\n    test_predictions_df = predict_for_test(model, test_loader, device, feature_spec)\n\n    metrics_output = {\n        "best_epoch": best_epoch,\n        "best_validation_metrics": best_metrics,\n        "final_validation_metrics": final_valid_metrics,\n        "selected_component_numeric_columns": (\n            feature_spec["component_numeric_columns"] + feature_spec["component_auto_numeric_columns"]\n        ),\n        "selected_component_categorical_columns": feature_spec["component_categorical_columns"],\n        "global_columns": feature_spec["global_columns"],\n        "family_count": len(feature_spec["family_vocabulary"]),\n        "history": history,\n    }\n\n    valid_predictions_df.to_csv(out_dir / "validation_predictions_hierarchical_model.csv", index=False)\n    test_predictions_df.to_csv(out_dir / "test_predictions_hierarchical_model.csv", index=False)\n    save_json(metrics_output, out_dir / "validation_metrics_hierarchical_model.json")\n\n    torch.save(\n        {\n            "model_state_dict": model.state_dict(),\n            "feature_spec": {\n                "component_numeric_columns": feature_spec["component_numeric_columns"],\n                "component_auto_numeric_columns": feature_spec["component_auto_numeric_columns"],\n                "component_categorical_columns": feature_spec["component_categorical_columns"],\n                "global_columns": feature_spec["global_columns"],\n                "category_vocabularies": feature_spec["category_vocabularies"],\n                "family_vocabulary": feature_spec["family_vocabulary"],\n                "component_numeric_mean": feature_spec["component_numeric_mean"].tolist(),\n                "component_numeric_std": feature_spec["component_numeric_std"].tolist(),\n                "global_mean": feature_spec["global_mean"].tolist(),\n                "global_std": feature_spec["global_std"].tolist(),\n                "target_mean": feature_spec["target_mean"].tolist(),\n                "target_std": feature_spec["target_std"].tolist(),\n            },\n        },\n        out_dir / "hierarchical_model.pt",\n    )\n\n    print("Обучение завершено.")\n    print(f"Лучшая эпоха: {best_epoch}")\n    print(json.dumps(best_metrics, ensure_ascii=False, indent=2))\n    print(f"Validation predictions: {(out_dir / \'validation_predictions_hierarchical_model.csv\').resolve()}")\n    print(f"Test predictions: {(out_dir / \'test_predictions_hierarchical_model.csv\').resolve()}")\n    print(f"Metrics: {(out_dir / \'validation_metrics_hierarchical_model.json\').resolve()}")\n    print(f"Model: {(out_dir / \'hierarchical_model.pt\').resolve()}")\n\n\nif __name__ == "__main__":\n    main()'


In [ ]:
from pathlib import Path
import shutil
import sys
import pandas as pd

root = Path.cwd()
train_path = root / "data" / "daimler_mixtures_train.csv"
test_path = root / "data" / "daimler_mixtures_test.csv"
props_path = root / "data" / "daimler_component_properties.csv"
runtime_dir = root / "_notebook_runtime_o2"
out_dir = runtime_dir / "train_out"
prediction_path = root / "prediction.csv"

for path in (train_path, test_path, props_path):
    if not path.exists():
        raise FileNotFoundError(path)

# Transform raw CSV -> component/scenario tables
transform_ns = {}
exec(TRANSFORM_SOURCE, transform_ns)
train_df = transform_ns['read_csv_strict'](train_path)
test_df = transform_ns['read_csv_strict'](test_path)
props_df = transform_ns['read_csv_strict'](props_path)
transform_ns['assert_required_columns'](train_df, transform_ns['REQUIRED_MIXTURE_COLS'] + transform_ns['TRAIN_TARGET_COLS'], 'train')
transform_ns['assert_required_columns'](test_df, transform_ns['REQUIRED_MIXTURE_COLS'], 'test')
transform_ns['assert_required_columns'](props_df, transform_ns['PROPS_REQUIRED_COLS'], 'properties')

for df in (train_df, test_df):
    df['scenario_id'] = df['scenario_id'].map(transform_ns['normalize_string'])
    df['Компонент'] = df['Компонент'].map(transform_ns['normalize_string'])
    df['Наименование партии'] = df['Наименование партии'].map(transform_ns['normalize_string'])

props_df['Компонент'] = props_df['Компонент'].map(transform_ns['normalize_string'])
props_df['Наименование партии'] = props_df['Наименование партии'].map(transform_ns['normalize_string'])
props_df['Наименование показателя'] = props_df['Наименование показателя'].map(transform_ns['normalize_property_name'])
props_df['Значение показателя'] = props_df['Значение показателя'].map(transform_ns['normalize_property_value'])
wide_exact, wide_typical, property_columns = transform_ns['build_property_tables'](props_df)
component_train_df, scenario_train_df = transform_ns['transform_dataset'](train_df, wide_exact, wide_typical, property_columns, True)
component_test_df, scenario_test_df = transform_ns['transform_dataset'](test_df, wide_exact, wide_typical, property_columns, False)

# Add O2 features
o2_train = build_o2_features(component_train_df, scenario_train_df)
o2_test = build_o2_features(component_test_df, scenario_test_df)
o2_columns = [
    'o2_salicylate_tbn_x_amine_ao',
    'o2_salicylate_tbn_x_phenol_ao',
    'o2_salicylate_tbn_x_amine_x_phenol',
    'o2_ca_salicylate_present',
    'o2_mg_detergent_present',
]
scenario_train_aug = merge_features(scenario_train_df, o2_train, o2_columns)
scenario_test_aug = merge_features(scenario_test_df, o2_test, o2_columns)

runtime_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)
component_train_path = runtime_dir / 'train_component_level_transformed.csv'
component_test_path = runtime_dir / 'test_component_level_transformed.csv'
scenario_train_path = runtime_dir / 'train_scenario_level_features_o2.csv'
scenario_test_path = runtime_dir / 'test_scenario_level_features_o2.csv'
component_train_df.to_csv(component_train_path, index=False)
component_test_df.to_csv(component_test_path, index=False)
scenario_train_aug.to_csv(scenario_train_path, index=False)
scenario_test_aug.to_csv(scenario_test_path, index=False)

# Run embedded trainer main() on generated CSV
trainer_ns = {}
exec(TRAINER_SOURCE, trainer_ns)
argv_backup = list(sys.argv)
sys.argv = [
    'inference.ipynb',
    '--component-train', str(component_train_path),
    '--component-test', str(component_test_path),
    '--scenario-train', str(scenario_train_path),
    '--scenario-test', str(scenario_test_path),
    '--out-dir', str(out_dir),
    '--seed', '42',
]
try:
    trainer_ns['main']()
finally:
    sys.argv = argv_backup

generated_prediction_path = out_dir / 'test_predictions_hierarchical_model.csv'
if prediction_path.exists():
    existing_prediction_df = pd.read_csv(prediction_path)
    generated_prediction_df = pd.read_csv(generated_prediction_path)
    if existing_prediction_df.shape == generated_prediction_df.shape and existing_prediction_df.columns.tolist() == generated_prediction_df.columns.tolist():
        print(f'Existing baseline prediction.csv preserved: {prediction_path.resolve()}')
        print(f'Fresh model output saved separately: {generated_prediction_path.resolve()}')
    else:
        shutil.copyfile(generated_prediction_path, prediction_path)
        print(f'prediction.csv saved to: {prediction_path.resolve()}')
else:
    shutil.copyfile(generated_prediction_path, prediction_path)
    print(f'prediction.csv saved to: {prediction_path.resolve()}')
